In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import re
import glob
import dask
from netCDF4 import Dataset, num2date
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import shutil
from datetime import datetime

GPM 전처리

In [ ]:
# 원본 폴더 경로
source_folder = r"C:\Yeonji\GPM"

# 새로 저장할 폴더 경로
target_folder = os.path.join(source_folder, "rename")

# 폴더 없으면 생성
os.makedirs(target_folder, exist_ok=True)

# 정규표현식: 날짜 추출
pattern = re.compile(r"\.3IMERG\.(\d{8})-S")

# 파일 순회
for filename in os.listdir(source_folder):
    match = pattern.search(filename)
    if match:
        date_str = match.group(1)
        new_name = f"{date_str}.nc4"

        src = os.path.join(source_folder, filename)
        dst = os.path.join(target_folder, new_name)

        try:
            shutil.copy2(src, dst)
            print(f"✔ 복사 완료: {filename} → {new_name}")
        except Exception as e:
            print(f"❌ 오류 발생: {filename} → {e}")

Station 전처리

In [ ]:
import os
import zipfile
import re
import shutil
import pandas as pd
import numpy as np
import glob

In [ ]:
os.chdir(r'C:\Yeonji\2025.01.Drought\2004')

In [ ]:
# 해당 폴더에 압축해제
def unzip_all_in_directory(directory='.'):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.zip'):
                zip_path = os.path.join(root, file)
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(root)
                print(f'{zip_path} has been extracted.')

In [ ]:
# 현재 디렉토리에서 실행
unzip_all_in_directory("0.StationRawdata")

In [ ]:
# 원본 파일이 위치한 디렉토리
source_dir = r"0.StationRawdata"

# 파일명에서 숫자 추출 및 해당 숫자 이름의 폴더로 파일 이동
for filename in os.listdir(source_dir):
    if filename.endswith(".csv"):
        # 파일명에서 숫자 추출 
        # (예: 'SURFACE_ASOS_187_DAY_2003_2003_2018.csv' 또는 'SURFACE_AWS_187_DAY_2003_2003_2018.csv' 에서 '187')
        match = re.search(r"_A(?:SOS|WS)_(\d+)_", filename)
        if match:
            folder_name = match.group(1)
            # 대상 폴더 경로 생성
            target_dir = os.path.join(source_dir, folder_name)
            if not os.path.exists(target_dir):
                os.makedirs(target_dir)
            # 파일 이동
            shutil.move(os.path.join(source_dir, filename), os.path.join(target_dir, filename))
            print(f"Moved {filename} to {target_dir}")

In [ ]:
base_folder_path = r'0.StationRawdata'
save_folder_path = r'1.Input_file'

# 저장할 폴더가 없으면 생성
if not os.path.exists(save_folder_path):
    os.makedirs(save_folder_path)

# 최상위 폴더 내의 모든 하위 폴더를 찾기
subfolders = [f.path for f in os.scandir(base_folder_path) if f.is_dir()]

# 각 하위 폴더에 대한 처리
for folder in subfolders:
    # 현재 폴더의 모든 CSV 파일을 찾기
    all_files = glob.glob(os.path.join(folder, "*.csv"))
    
    # 각 파일에서 필요한 데이터를 추출하여 하나의 DataFrame으로 합치기
    all_data = []
    for file in all_files:
        df = pd.read_csv(file, usecols=['일시', '일강수량(mm)'], encoding='CP949')
        # 열 이름 변경
        df.rename(columns={'일시': 'Date', '일강수량(mm)': 'Precipitation'}, inplace=True)
        all_data.append(df)
    
    if all_data:
        merged_data = pd.concat(all_data, ignore_index=True)

        # 날짜 열을 기준으로 데이터 정렬
        merged_data['Date'] = pd.to_datetime(merged_data['Date'])
        merged_data = merged_data.sort_values(by='Date')

        # 결과를 지정된 폴더에 저장 (파일 이름은 폴더명_Daily_Precipitation.csv)
        output_file_name = os.path.basename(folder) + '_Daily_Precipitation.csv'
        output_file_path = os.path.join(save_folder_path, output_file_name)
        merged_data.to_csv(output_file_path, index=False)

In [ ]:
# Nan 값 0으로 채우기
# 폴더 경로 설정
folder_path = '1.Input_file'

# 모든 CSV 파일을 찾기
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

# 각 파일에 대한 처리
for file in all_files:
    # 파일 불러오기
    df = pd.read_csv(file)

    # 'Precipitation' 열에서 빈 값들을 0으로 대체
    df['Precipitation'] = df['Precipitation'].fillna(0)

    # 파일 저장
    df.to_csv(file, index=False)

In [ ]:
# 날짜 범위 설정
start_date = '2004-01-01'
end_date = '2023-12-31'
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# 파일 경로 설정
folder_path = r'1.Input_file'

# 누락된 날짜 정보를 저장할 DataFrame 생성
missing_dates_summary = pd.DataFrame(columns=['File', 'Missing Date'])

# 폴더 내의 모든 파일 목록 가져오기 (예: CSV 파일을 찾는 경우)
file_pattern = os.path.join(folder_path, '*.csv')  # 파일 확장자에 맞게 수정하세요
file_list = glob.glob(file_pattern)

# 각 파일에 대해 처리
for file_path in file_list:
    filename = os.path.basename(file_path)
    print(f"처리 중: {filename}")
    
    try:
        # 파일 읽기 (파일 형식에 맞게 수정하세요)
        df = pd.read_csv(file_path)
        
        # 날짜 열 이름을 파일 형식에 맞게 수정하세요
        date_column = 'Date'  # 예: 'Date', 'Timestamp', '날짜' 등
        
        # 날짜 열이 문자열인 경우 datetime으로 변환
        if date_column in df.columns:
            df[date_column] = pd.to_datetime(df[date_column])
        else:
            print(f"경고: {filename}에 '{date_column}' 열이 없습니다. 다음 파일로 넘어갑니다.")
            continue
        
        # 파일의 날짜 목록
        file_dates = set(df[date_column].dt.date)
        
        # 전체 날짜 범위와 비교하여 누락된 날짜 찾기
        expected_dates = set(date_range.date)
        missing_dates = expected_dates - file_dates
        
        # 누락된 날짜가 있으면 결과에 추가
        if missing_dates:
            for date in sorted(missing_dates):
                # 추가할 새로운 행을 DataFrame으로 만듦
                new_row = pd.DataFrame({'File': [filename], 'Missing Date': [date]})
                
                # pd.concat으로 새로운 데이터를 추가
                missing_dates_summary = pd.concat([missing_dates_summary, new_row], ignore_index=True)
            
            print(f"  {filename}에서 {len(missing_dates)}개의 누락된 날짜를 찾았습니다.")
        else:
            print(f"  {filename}에서 누락된 날짜가 없습니다.")
    
    except Exception as e:
        print(f"오류 발생: {filename} 처리 중 - {e}")

# 결과를 Excel 파일로 저장
excel_path = os.path.join(folder_path, 'Missing_Dates_Summary.xlsx')
missing_dates_summary.to_excel(excel_path, index=False)
print(f"\n결과가 저장되었습니다: {excel_path}")
print(f"총 {len(missing_dates_summary)}개의 누락된 날짜 항목이 발견되었습니다.")

In [ ]:
def process_precipitation_files(input_folder_path, output_folder_path):
    # 출력 폴더가 없으면 생성
    if not os.path.exists(output_folder_path):
        os.makedirs(output_folder_path)
        
    for filename in os.listdir(input_folder_path):
        if filename.endswith('.csv'):
            input_file_path = os.path.join(input_folder_path, filename)
            
            try:
                print(f"Processing: {filename}")
                
                # Read the CSV file
                df = pd.read_csv(input_file_path)
                
                # 파일 내용 확인
                print(f"Columns in the file: {df.columns.tolist()}")
                print(f"First few rows:\n{df.head()}")

                # Remove duplicates if 'Date' column exists
                if 'Date' in df.columns:
                    df = df.drop_duplicates(subset=['Date'])
                    # Convert 'Date' column to datetime format
                    df['Date'] = pd.to_datetime(df['Date'])
                else:
                    print(f"Warning: 'Date' column not found in {filename}")
                    continue
                
                # 중간 결과 저장을 위한 새 DataFrame 생성
                result_df = pd.DataFrame()
                
                # 월별 데이터 생성 (직접 계산)
                years = df['Date'].dt.year.unique()
                
                for year in years:
                    year_data = df[df['Date'].dt.year == year]
                    months = year_data['Date'].dt.month.unique()
                    
                    for month in months:
                        month_data = year_data[year_data['Date'].dt.month == month]
                        monthly_sum = month_data['Precipitation'].sum()
                        
                        # 새 행 추가
                        date_str = f"{year}-{month:02d}-01"
                        new_row = pd.DataFrame({
                            'Date': [pd.to_datetime(date_str)],
                            'Precipitation': [monthly_sum]
                        })
                        
                        result_df = pd.concat([result_df, new_row], ignore_index=True)
                
                # 결과 정렬
                result_df = result_df.sort_values('Date').reset_index(drop=True)
                
                # Change the file name from 'Daily' to 'Monthly'
                output_filename = filename.replace('Daily', 'Monthly')

                # Save the monthly precipitation data to a new CSV file in the output folder
                output_file_path = os.path.join(output_folder_path, output_filename)
                result_df.to_csv(output_file_path, index=False)
                print(f"Successfully processed and saved: {output_filename}")
                print(f"Result has {len(result_df)} rows with columns: {result_df.columns.tolist()}")
                
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
                # 오류 세부 정보 출력
                import traceback
                traceback.print_exc()


# 실행 예시
if __name__ == "__main__":
    input_folder = "1.Input_file"
    output_folder = "1.Input_file/2.Monthly_Precipitation"
    
    process_precipitation_files(input_folder, output_folder)